In [6]:
import sys
sys.path.append("/home/kp_info_loader")


from exchange_plugin.okx_plug import InitOkxAdaptor
from exchange_plugin.upbit_plug import InitUpbitAdaptor
from exchange_plugin.binance_plug import InitBinanceAdaptor
from exchange_plugin.bithumb_plug import InitBithumbAdaptor
from exchange_plugin.bybit_plug import InitBybitAdaptor
from etc.db_handler.mongodb_client import InitDBClient # TEMP
from etc.register_monitor_msg import RegisterMonitorMsg # TEMP
import pandas as pd
import json
import time
import datetime
from threading import Thread
import numpy as np

info_dict = {}
info_thread_dict = {}

okx_adaptor = InitOkxAdaptor("fcb66a6e-0443-4743-8d9b-61caf7eab662", "0029E09874B38F5AC7E68E9DFC348667", "121431Rn!!")
upbit_adaptor = InitUpbitAdaptor("x2AKLfsRtAgRxFjQ7p9TZTAg6E1SveoqfNfD8MK8", "svEran52QdsUyJdxAPYoTgFT2MXsHc5ZiKsKPxTL", info_dict, None)
binance_adaptor = InitBinanceAdaptor("4PFfYsObYyUaMlk7m6tT9qIl8X8P3WCUsyu1lEyZ4h50VfTMPIQCNdL9eOJCl3Lb", "011Z1W9p4425AZgtCNbs5L1d77ehZFNmBIwT0pz1SJGP7EiOlPILfWBglbVsxMcK", info_dict, None)
bithumb_adaptor = InitBithumbAdaptor()
bybit_adaptor = InitBybitAdaptor()

# UPBIT SPOT (KRW, BTC Market)
# UPBIT wallet status
# BINANCE SPOT, USD-M Futures, COIN-M Futures
data_name_list = [
        "upbit_spot_info_df",
        "upbit_spot_ticker_df",
        # "upbit_wallet_status_df",
        "binance_spot_ticker_df",
        "binance_spot_info_df",
        "binance_usd_m_ticker_df",
        "binance_usd_m_info_df",
        "binance_coin_m_ticker_df",
        "binance_coin_m_info_df",
        "okx_spot_ticker_df",
        "okx_spot_info_df",
        "okx_usd_m_ticker_df",
        "okx_usd_m_info_df",
        "okx_coin_m_ticker_df",
        "okx_coin_m_info_df",
        "bithumb_spot_info_df",
        "bithumb_spot_ticker_df",
        # "bithumb_wallet_status_df",
        "bybit_spot_info_df",
        "bybit_spot_ticker_df",
        "bybit_usd_m_info_df",
        "bybit_usd_m_ticker_df",
        "bybit_coin_m_info_df",
        "bybit_coin_m_ticker_df"
    ]

def update_exchange_info_as_df(data_name, loop_time_secs=15):
    if data_name == "upbit_spot_ticker_df":
        info_dict[data_name] = upbit_adaptor.spot_all_tickers()
    elif data_name == "upbit_wallet_status_df":
        info_dict[data_name] = upbit_adaptor.wallet_status()
    elif data_name == "upbit_spot_info_df":
        info_dict[data_name] = upbit_adaptor.spot_exchange_info()
    elif data_name == "binance_spot_ticker_df":
        info_dict[data_name] = binance_adaptor.spot_all_tickers()
    elif data_name == "binance_spot_info_df":
        info_dict[data_name] = binance_adaptor.spot_exchange_info()
    elif data_name == "binance_usd_m_ticker_df":
        info_dict[data_name] = binance_adaptor.usd_m_all_tickers()
    elif data_name == "binance_usd_m_info_df":
        info_dict[data_name] = binance_adaptor.usd_m_exchange_info()
    elif data_name == "binance_coin_m_ticker_df":
        info_dict[data_name] = binance_adaptor.coin_m_all_tickers()
    elif data_name == "binance_coin_m_info_df":
        info_dict[data_name] = binance_adaptor.coin_m_exchange_info()
    elif data_name == "okx_spot_ticker_df":
        info_dict[data_name] = okx_adaptor.spot_all_tickers()
    elif data_name == "okx_spot_info_df":
        info_dict[data_name] = okx_adaptor.spot_exchange_info()
    elif data_name == "okx_usd_m_ticker_df":
        info_dict[data_name] = okx_adaptor.usd_m_all_tickers()
    elif data_name == "okx_usd_m_info_df":
        info_dict[data_name] = okx_adaptor.usd_m_exchange_info()
    elif data_name == "okx_coin_m_ticker_df":
        info_dict[data_name] = okx_adaptor.coin_m_all_tickers()
    elif data_name == "okx_coin_m_info_df":
        info_dict[data_name] = okx_adaptor.coin_m_exchange_info()
    elif data_name == "bithumb_spot_info_df":
        info_dict[data_name] = bithumb_adaptor.spot_exchange_info()
    elif data_name == "bithumb_spot_ticker_df":
        info_dict[data_name] = bithumb_adaptor.spot_all_tickers()
    elif data_name == "bithumb_wallet_status_df":
        info_dict[data_name] = bithumb_adaptor.wallet_status()
    elif data_name == "bybit_spot_info_df":
        info_dict[data_name] = bybit_adaptor.spot_exchange_info()
    elif data_name == "bybit_spot_ticker_df":
        info_dict[data_name] = bybit_adaptor.spot_all_tickers()
    elif data_name == "bybit_usd_m_info_df":
        info_dict[data_name] = bybit_adaptor.usd_m_exchange_info()
    elif data_name == "bybit_usd_m_ticker_df":
        info_dict[data_name] = bybit_adaptor.usd_m_all_tickers()
    elif data_name == "bybit_coin_m_info_df":
        info_dict[data_name] = bybit_adaptor.coin_m_exchange_info()
    elif data_name == "bybit_coin_m_ticker_df":
        info_dict[data_name] = bybit_adaptor.coin_m_all_tickers()
    else:
        print(f"update_exchange_info_as_df|name:{data_name} is not valid.")

for data_name in data_name_list:
    info_thread_dict[f"update_{data_name}"] = Thread(target=update_exchange_info_as_df, args=(data_name,), daemon=True)
    info_thread_dict[f"update_{data_name}"].start()
    print(f"InitCore|update_{data_name} thread has started.")


enabled_markets_dict = \
    {
        "UPBIT": 
        {
            "SPOT": ["KRW","BTC"]
        },
        "BITHUMB":
        {
            "SPOT": ["KRW"]
        },
        "BINANCE":
        {
            "SPOT": ["USDT", "BTC"],
            "USD_M": {
                "PERPETUAL": ["USDT"],
                "FUTURES": []
            },
            "COIN_M": {
                "PERPETUAL": ["USD"],
                "FUTURES": []
            }
        },
        "OKX":
        {
            "SPOT": ["USDT", "BTC"],
            "USD_M": {
                "PERPETUAL": ["USDT"],
                "FUTURES": []
            },
            "COIN_M": {
                "PERPETUAL": ["USD"],
                "FUTURES": []
            }
        },
        "BYBIT":
        {
            "SPOT": ["USDT"],
            "USD_M": {
                "PERPETUAL": ["USDT"],
                "FUTURES": []
            },
            "COIN_M": {
                "PERPETUAL": ["USD"],
                "FUTURES": []
            }
        }
    }


with open("/home/kp_info_loader/kp_info_loader_config.json") as f:
        config = json.load(f)

node = config['node']
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id_list = []
admin_id = config['telegram_admin_id']['charlie1155']
admin_id_list.append(admin_id)
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir=None)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
enabled_markets_dict = config['enabled_markets']

db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

2023-11-10 14:11:02,061 - okx_plug - INFO - okx_plug_logger started.
2023-11-10 14:11:02,061 - okx_plug - INFO - okx_plug_logger started.
2023-11-10 14:11:02,061 - okx_plug - INFO - okx_plug_logger started.
2023-11-10 14:11:02,061 - okx_plug - INFO - okx_plug_logger started.
2023-11-10 14:11:02,586 - upbit_plug - INFO - upbit_plug_logger started.
2023-11-10 14:11:02,586 - upbit_plug - INFO - upbit_plug_logger started.
2023-11-10 14:11:02,586 - upbit_plug - INFO - upbit_plug_logger started.
2023-11-10 14:11:02,586 - upbit_plug - INFO - upbit_plug_logger started.
2023-11-10 14:11:02,707 - binance_plug - INFO - binance_plug_logger started.
2023-11-10 14:11:02,707 - binance_plug - INFO - binance_plug_logger started.
2023-11-10 14:11:02,707 - binance_plug - INFO - binance_plug_logger started.
2023-11-10 14:11:02,707 - binance_plug - INFO - binance_plug_logger started.
2023-11-10 14:11:02,740 - bithumb_plug - INFO - bithumb_plug_logger started.
2023-11-10 14:11:02,740 - bithumb_plug - INFO -

InitCore|update_upbit_spot_info_df thread has started.
InitCore|update_upbit_spot_ticker_df thread has started.
InitCore|update_binance_spot_ticker_df thread has started.
InitCore|update_binance_spot_info_df thread has started.
InitCore|update_binance_usd_m_ticker_df thread has started.
InitCore|update_binance_usd_m_info_df thread has started.
InitCore|update_binance_coin_m_ticker_df thread has started.
InitCore|update_binance_coin_m_info_df thread has started.
InitCore|update_okx_spot_ticker_df thread has started.
InitCore|update_okx_spot_info_df thread has started.
InitCore|update_okx_usd_m_ticker_df thread has started.
InitCore|update_okx_usd_m_info_df thread has started.
InitCore|update_okx_coin_m_ticker_df thread has started.
InitCore|update_okx_coin_m_info_df thread has started.
InitCore|update_bithumb_spot_info_df thread has started.
InitCore|update_bithumb_spot_ticker_df thread has started.
InitCore|update_bybit_spot_info_df thread has started.
InitCore|update_bybit_spot_ticker

2023-11-10 14:11:03,436 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_coin_m_info_df') is None, Fetched from API
2023-11-10 14:11:03,436 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_coin_m_info_df') is None, Fetched from API
2023-11-10 14:11:03,436 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_coin_m_info_df') is None, Fetched from API
2023-11-10 14:11:03,436 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_coin_m_info_df') is None, Fetched from API
2023-11-10 14:11:05,395 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_spot_info_df') is None, Fetched from API
2023-11-10 14:11:05,396 - binance_plug - INFO - self.info_dict is None or self.info_dict.get('binance_usd_m_info_df') is None, Fetched from API
2023-11-10 14:11:05,396 - binance_plug - INFO - self.info_dict is None or self.info_dict.get('binance_usd_m_info_df') is None, Fetched from API
2023-11-1

In [8]:
enabled_markets_dict

{'UPBIT': {'SPOT': ['KRW', 'BTC']},
 'BITHUMB': {'SPOT': ['KRW']},
 'BINANCE': {'SPOT': ['USDT', 'BTC'],
  'USD_M': {'PERPETUAL': ['USDT'], 'FUTURES': []},
  'COIN_M': {'PERPETUAL': ['USD'], 'FUTURES': []}},
 'OKX': {'SPOT': ['USDT', 'BTC'],
  'USD_M': {'PERPETUAL': ['USDT'], 'FUTURES': []},
  'COIN_M': {'PERPETUAL': ['USD'], 'FUTURES': []}},
 'BYBIT': {'SPOT': ['USDT'],
  'USD_M': {'PERPETUAL': ['USDT'], 'FUTURES': []},
  'COIN_M': {'PERPETUAL': ['USD'], 'FUTURES': []}}}

In [10]:
enabled_markets_dict.keys()

dict_keys(['UPBIT', 'BITHUMB', 'BINANCE', 'OKX', 'BYBIT'])

In [15]:
perpetual_tup_list = []

for exchange in enabled_markets_dict.keys():
    if "USD_M" not in enabled_markets_dict[exchange].keys() and "COIN_M" not in enabled_markets_dict[exchange].keys():
        continue
    for market_type in enabled_markets_dict[exchange].keys():
        if market_type == "SPOT":
            continue
        for contract_type in enabled_markets_dict[exchange][market_type].keys():
            if contract_type == "FUTURES":
                continue
            
            for quote_asset in enabled_markets_dict[exchange][market_type][contract_type]:
                perpetual_tup_list.append((exchange, market_type, quote_asset))

exchange: BINANCE, market_type: USD_M, contract_type: PERPETUAL, quote_asset: USDT
exchange: BINANCE, market_type: COIN_M, contract_type: PERPETUAL, quote_asset: USD
exchange: OKX, market_type: USD_M, contract_type: PERPETUAL, quote_asset: USDT
exchange: OKX, market_type: COIN_M, contract_type: PERPETUAL, quote_asset: USD
exchange: BYBIT, market_type: USD_M, contract_type: PERPETUAL, quote_asset: USDT
exchange: BYBIT, market_type: COIN_M, contract_type: PERPETUAL, quote_asset: USD


In [88]:
perpetual_tup_list

[('BINANCE', 'USD_M', 'USDT'),
 ('BINANCE', 'COIN_M', 'USD'),
 ('OKX', 'USD_M', 'USDT'),
 ('OKX', 'COIN_M', 'USD'),
 ('BYBIT', 'USD_M', 'USDT'),
 ('BYBIT', 'COIN_M', 'USD')]

In [20]:
perpetual_combination_list = []
for i, perpetual_tup_one in enumerate(perpetual_tup_list):
    for perpetual_tup_two in perpetual_tup_list[i+1:]:
        perpetual_combination_list.append((perpetual_tup_one, perpetual_tup_two))

perpetual_combination_list

[(('BINANCE', 'USD_M', 'USDT'), ('BINANCE', 'COIN_M', 'USD')),
 (('BINANCE', 'USD_M', 'USDT'), ('OKX', 'USD_M', 'USDT')),
 (('BINANCE', 'USD_M', 'USDT'), ('OKX', 'COIN_M', 'USD')),
 (('BINANCE', 'USD_M', 'USDT'), ('BYBIT', 'USD_M', 'USDT')),
 (('BINANCE', 'USD_M', 'USDT'), ('BYBIT', 'COIN_M', 'USD')),
 (('BINANCE', 'COIN_M', 'USD'), ('OKX', 'USD_M', 'USDT')),
 (('BINANCE', 'COIN_M', 'USD'), ('OKX', 'COIN_M', 'USD')),
 (('BINANCE', 'COIN_M', 'USD'), ('BYBIT', 'USD_M', 'USDT')),
 (('BINANCE', 'COIN_M', 'USD'), ('BYBIT', 'COIN_M', 'USD')),
 (('OKX', 'USD_M', 'USDT'), ('OKX', 'COIN_M', 'USD')),
 (('OKX', 'USD_M', 'USDT'), ('BYBIT', 'USD_M', 'USDT')),
 (('OKX', 'USD_M', 'USDT'), ('BYBIT', 'COIN_M', 'USD')),
 (('OKX', 'COIN_M', 'USD'), ('BYBIT', 'USD_M', 'USDT')),
 (('OKX', 'COIN_M', 'USD'), ('BYBIT', 'COIN_M', 'USD')),
 (('BYBIT', 'USD_M', 'USDT'), ('BYBIT', 'COIN_M', 'USD'))]

In [92]:

def get_funding_combination(enabled_markets_dict, head=None, same_exchange=False):
    total_df = pd.DataFrame()
    db_client = InitDBClient(**db_dict)
    mongo_db_conn = db_client.get_conn()

    perpetual_tup_list = []
    for exchange in enabled_markets_dict.keys():
        if "USD_M" not in enabled_markets_dict[exchange].keys() and "COIN_M" not in enabled_markets_dict[exchange].keys():
            continue
        for market_type in enabled_markets_dict[exchange].keys():
            if market_type == "SPOT":
                continue
            for contract_type in enabled_markets_dict[exchange][market_type].keys():
                if contract_type == "FUTURES":
                    continue
                
                for quote_asset in enabled_markets_dict[exchange][market_type][contract_type]:
                    perpetual_tup_list.append((exchange, market_type, quote_asset))

    perpetual_combination_list = []
    for i, perpetual_tup_one in enumerate(perpetual_tup_list):
        for perpetual_tup_two in perpetual_tup_list[i+1:]:
            perpetual_combination_list.append((perpetual_tup_one, perpetual_tup_two))

    for perpetual_combination in perpetual_combination_list:
        perpetual_one = perpetual_combination[0]
        perpetual_two = perpetual_combination[1]

        mongo_db = mongo_db_conn[f"{perpetual_one[0]}_fundingrate"]
        mongo_db_collection = mongo_db[f"{perpetual_one[1]}"]

        pipeline = [
            {"$match": {"quote_asset": perpetual_one[2], "perpetual":True}},
            {"$sort": {"datetime_now": -1}},
            {"$group": {
                "_id": "$symbol",  # Group by symbol
                "latest_document": {"$first": "$$ROOT"}  # Get the first document after sorting (i.e., the latest)
            }}
        ]

        # Execute the aggregation pipeline
        result = list(mongo_db_collection.aggregate(pipeline))

        # Now `result` will have the latest document for each symbol with 'quote_asset'
        # Convert the result into a DataFrame
        # The latest documents are under 'latest_document' in the result
        latest_documents = [doc['latest_document'] for doc in result]
        funding_df1 = pd.DataFrame(latest_documents).drop(columns=["_id", "perpetual"])
        funding_df1['market_code'] = f"{perpetual_one[0]}_{perpetual_one[1]}"

        # For funding_df2
        mongo_db = mongo_db_conn[f"{perpetual_two[0]}_fundingrate"]
        mongo_db_collection = mongo_db[f"{perpetual_two[1]}"]

        pipeline = [
            {"$match": {"quote_asset": perpetual_two[2], "perpetual":True}},
            {"$sort": {"datetime_now": -1}},
            {"$group": {
                "_id": "$symbol",  # Group by symbol
                "latest_document": {"$first": "$$ROOT"}  # Get the first document after sorting (i.e., the latest)
            }}
        ]

        # Execute the aggregation pipeline
        result = list(mongo_db_collection.aggregate(pipeline))

        # Now `result` will have the latest document for each symbol with 'quote_asset'
        # Convert the result into a DataFrame
        # The latest documents are under 'latest_document' in the result
        latest_documents = [doc['latest_document'] for doc in result]
        funding_df2 = pd.DataFrame(latest_documents).drop(columns=["_id", "perpetual"])
        funding_df2['market_code'] = f"{perpetual_two[0]}_{perpetual_two[1]}"

        merged_df = funding_df1.merge(funding_df2, on='base_asset', how='inner')
        total_df = pd.concat([total_df, merged_df], axis=0)
        total_df['funding_rate_diff'] = (total_df['funding_rate_x'] - total_df['funding_rate_y']).abs()
        total_df = total_df.sort_values(by=['funding_rate_diff'], ascending=False).reset_index(drop=True)
        if same_exchange:
            total_df = total_df[total_df['market_code_x'].str.split('_').str[0]==total_df['market_code_y'].str.split('_').str[0]].reset_index(drop=True)
        if head is not None:
            total_df = total_df.head(head)
    mongo_db_conn.close()
    return total_df

def get_all_funding_rate(enabled_markets_dict):
    total_df = pd.DataFrame()
    db_client = InitDBClient(**db_dict)
    mongo_db_conn = db_client.get_conn()

    perpetual_tup_list = []
    for exchange in enabled_markets_dict.keys():
        if "USD_M" not in enabled_markets_dict[exchange].keys() and "COIN_M" not in enabled_markets_dict[exchange].keys():
            continue
        for market_type in enabled_markets_dict[exchange].keys():
            if market_type == "SPOT":
                continue
            for contract_type in enabled_markets_dict[exchange][market_type].keys():
                if contract_type == "FUTURES":
                    continue
                
                for quote_asset in enabled_markets_dict[exchange][market_type][contract_type]:
                    perpetual_tup_list.append((exchange, market_type, quote_asset))

    for perpetual_tup in perpetual_tup_list:
        exchange = perpetual_tup[0]
        market_type = perpetual_tup[1]
        quote_asset = perpetual_tup[2]
        mongo_db = mongo_db_conn[f"{exchange}_fundingrate"]
        mongo_db_collection = mongo_db[f"{market_type}"]

        pipeline = [
            {"$match": {"quote_asset": quote_asset, "perpetual":True}},
            {"$sort": {"datetime_now": -1}},
            {"$group": {
                "_id": "$symbol",  # Group by symbol
                "latest_document": {"$first": "$$ROOT"}  # Get the first document after sorting (i.e., the latest)
            }}
        ]

        # Execute the aggregation pipeline
        result = list(mongo_db_collection.aggregate(pipeline))

        # Now `result` will have the latest document for each symbol with 'quote_asset'
        # Convert the result into a DataFrame
        # The latest documents are under 'latest_document' in the result
        latest_documents = [doc['latest_document'] for doc in result]
        funding_df = pd.DataFrame(latest_documents).drop(columns=["_id", "perpetual"])
        funding_df['market_code'] = f"{exchange}_{market_type}"
        total_df = pd.concat([total_df, funding_df], axis=0)
    
    total_df = total_df.sort_values('funding_rate').reset_index(drop=True)
    mongo_db_conn.close()
    return total_df

    


In [94]:
temp = get_all_funding_rate(enabled_markets_dict)
temp

,symbol,funding_rate,funding_time,base_asset,quote_asset,datetime_now,market_code
0,TIAUSDT,0.000366,2023-11-10 08:00:00,TIA,USDT,2023-11-10 07:35:07.263,BINANCE_USD_M
1,TLMUSDT,0.000100,2023-11-10 08:00:00,TLM,USDT,2023-11-10 07:35:07.263,BINANCE_USD_M
2,RVNUSDT,0.000110,2023-11-10 08:00:00,RVN,USDT,2023-11-10 07:35:07.263,BINANCE_USD_M
3,CHZUSDT,0.000100,2023-11-10 08:00:00,CHZ,USDT,2023-11-10 07:35:07.263,BINANCE_USD_M
4,ZILUSDT,0.000100,2023-11-10 08:00:00,ZIL,USDT,2023-11-10 07:35:07.263,BINANCE_USD_M
...,...,...,...,...,...,...,...
731,DOTUSD,0.000100,2023-11-10 08:00:00,DOT,USD,2023-11-10 07:34:56.404,BYBIT_COIN_M
732,EOSUSD,0.000100,2023-11-10 08:00:00,EOS,USD,2023-11-10 07:34:56.404,BYBIT_COIN_M
733,MANAUSD,0.000100,2023-11-10 08:00:00,MANA,USD,2023-11-10 07:34:56.404,BYBIT_COIN_M
734,XRPUSD,0.000100,2023-11-10 08:00:00,XRP,USD,2023-11-10 07:34:56.404,BYBIT_COIN_M


In [96]:
temp.sort_values('funding_rate', ascending=False)

,symbol,funding_rate,funding_time,base_asset,quote_asset,datetime_now,market_code
670,1000BONKUSDT,0.002500,2023-11-10 08:00:00,1000BONK,USDT,2023-11-10 07:34:49.532,BYBIT_USD_M
636,KASUSDT,0.001536,2023-11-10 08:00:00,KAS,USDT,2023-11-10 07:34:49.532,BYBIT_USD_M
559,1000LUNCUSDT,0.001427,2023-11-10 08:00:00,1000LUNC,USDT,2023-11-10 07:34:49.532,BYBIT_USD_M
473,FITFIUSDT,0.001397,2023-11-10 08:00:00,FITFI,USDT,2023-11-10 07:34:49.532,BYBIT_USD_M
442,THETA-USD-SWAP,0.001184,2023-11-10 08:00:00,THETA,USD,2023-11-10 07:33:46.261,OKX_COIN_M
...,...,...,...,...,...,...,...
223,TRBUSDT,-0.002239,2023-11-10 08:00:00,TRB,USDT,2023-11-10 07:35:07.263,BINANCE_USD_M
712,TRBUSDT,-0.002419,2023-11-10 08:00:00,TRB,USDT,2023-11-10 07:34:49.532,BYBIT_USD_M
296,GAS-USDT-SWAP,-0.008257,2023-11-10 08:00:00,GAS,USDT,2023-11-10 07:33:33.367,OKX_USD_M
534,GASUSDT,-0.020000,2023-11-10 08:00:00,GAS,USDT,2023-11-10 07:34:49.532,BYBIT_USD_M


In [95]:
temp['market_code'].unique()

array(['BINANCE_USD_M', 'BINANCE_COIN_M', 'OKX_USD_M', 'OKX_COIN_M',
       'BYBIT_USD_M', 'BYBIT_COIN_M'], dtype=object)

In [89]:
perpetual_tup_list

[('BINANCE', 'USD_M', 'USDT'),
 ('BINANCE', 'COIN_M', 'USD'),
 ('OKX', 'USD_M', 'USDT'),
 ('OKX', 'COIN_M', 'USD'),
 ('BYBIT', 'USD_M', 'USDT'),
 ('BYBIT', 'COIN_M', 'USD')]

In [87]:
get_funding_combination(head=30, same_exchange=True)

,symbol_x,funding_rate_x,funding_time_x,base_asset,quote_asset_x,datetime_now_x,market_code_x,symbol_y,funding_rate_y,funding_time_y,quote_asset_y,datetime_now_y,market_code_y,funding_rate_diff
0,THETA-USDT-SWAP,0.000238,2023-11-10 08:00:00,THETA,USDT,2023-11-10 07:24:43.326,OKX_USD_M,THETA-USD-SWAP,0.001184,2023-11-10 08:00:00,USD,2023-11-10 07:25:03.330,OKX_COIN_M,0.000946
1,ETHUSDT,0.000708,2023-11-10 08:00:00,ETH,USDT,2023-11-10 07:28:12.646,BYBIT_USD_M,ETHUSD,0.000140,2023-11-10 08:00:00,USD,2023-11-10 07:28:18.198,BYBIT_COIN_M,0.000568
2,DASH-USDT-SWAP,0.000117,2023-11-10 08:00:00,DASH,USDT,2023-11-10 07:24:43.326,OKX_USD_M,DASH-USD-SWAP,0.000574,2023-11-10 08:00:00,USD,2023-11-10 07:25:03.330,OKX_COIN_M,0.000457
3,EOS-USDT-SWAP,-0.000086,2023-11-10 08:00:00,EOS,USDT,2023-11-10 07:24:43.326,OKX_USD_M,EOS-USD-SWAP,0.000362,2023-11-10 08:00:00,USD,2023-11-10 07:25:03.330,OKX_COIN_M,0.000447
4,XRPUSDT,0.000515,2023-11-10 08:00:00,XRP,USDT,2023-11-10 07:28:12.646,BYBIT_USD_M,XRPUSD,0.000100,2023-11-10 08:00:00,USD,2023-11-10 07:28:18.198,BYBIT_COIN_M,0.000415
5,LTC-USDT-SWAP,0.000099,2023-11-10 08:00:00,LTC,USDT,2023-11-10 07:24:43.326,OKX_USD_M,LTC-USD-SWAP,0.000506,2023-11-10 08:00:00,USD,2023-11-10 07:25:03.330,OKX_COIN_M,0.000407
6,GALAUSDT,0.000433,2023-11-10 08:00:00,GALA,USDT,2023-11-10 07:28:21.469,BINANCE_USD_M,GALAUSD_PERP,0.000100,2023-11-10 08:00:00,USD,2023-11-10 07:28:28.881,BINANCE_COIN_M,0.000333
7,ADAUSDT,0.000239,2023-11-10 08:00:00,ADA,USDT,2023-11-10 07:28:12.646,BYBIT_USD_M,ADAUSD,0.000570,2023-11-10 08:00:00,USD,2023-11-10 07:28:18.198,BYBIT_COIN_M,0.000331
8,AVAX-USDT-SWAP,-0.000016,2023-11-10 08:00:00,AVAX,USDT,2023-11-10 07:24:43.326,OKX_USD_M,AVAX-USD-SWAP,0.000284,2023-11-10 08:00:00,USD,2023-11-10 07:25:03.330,OKX_COIN_M,0.000300
9,XRP-USDT-SWAP,-0.000302,2023-11-10 08:00:00,XRP,USDT,2023-11-10 07:24:43.326,OKX_USD_M,XRP-USD-SWAP,-0.000005,2023-11-10 08:00:00,USD,2023-11-10 07:25:03.330,OKX_COIN_M,0.000297


In [66]:
funding_df1

,symbol,funding_rate,funding_time,base_asset,quote_asset,datetime_now,market_code
0,MAGICUSDT,0.000158,2023-11-10 08:00:00,MAGIC,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M
1,LDOUSDT,0.000378,2023-11-10 08:00:00,LDO,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M
2,ASTRUSDT,0.000246,2023-11-10 08:00:00,ASTR,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M
3,DOTUSDT,0.000106,2023-11-10 08:00:00,DOT,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M
4,ARUSDT,0.000100,2023-11-10 08:00:00,AR,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M
...,...,...,...,...,...,...,...
267,ENJUSDT,-0.000460,2023-11-10 08:00:00,ENJ,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M
268,MKRUSDT,0.000100,2023-11-10 08:00:00,MKR,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M
269,MATICUSDT,0.000561,2023-11-10 08:00:00,MATIC,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M
270,INJUSDT,0.000100,2023-11-10 08:00:00,INJ,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M


In [67]:
funding_df2

,symbol,funding_rate,funding_time,base_asset,quote_asset,datetime_now,market_code
0,EOSUSD,0.000100,2023-11-10 08:00:00,EOS,USD,2023-11-10 07:09:30.679,BYBIT_COIN_M
1,MANAUSD,0.000100,2023-11-10 08:00:00,MANA,USD,2023-11-10 07:09:30.679,BYBIT_COIN_M
2,BTCUSD,0.000425,2023-11-10 08:00:00,BTC,USD,2023-11-10 07:09:30.679,BYBIT_COIN_M
3,ETHUSD,0.000134,2023-11-10 08:00:00,ETH,USD,2023-11-10 07:09:30.679,BYBIT_COIN_M
4,XRPUSD,0.000100,2023-11-10 08:00:00,XRP,USD,2023-11-10 07:09:30.679,BYBIT_COIN_M
5,LTCUSD,0.000371,2023-11-10 08:00:00,LTC,USD,2023-11-10 07:09:30.679,BYBIT_COIN_M
6,ADAUSD,0.000562,2023-11-10 08:00:00,ADA,USD,2023-11-10 07:09:30.679,BYBIT_COIN_M
7,DOTUSD,0.000100,2023-11-10 08:00:00,DOT,USD,2023-11-10 07:09:30.679,BYBIT_COIN_M


In [68]:
total_df['funding_rate_diff'] = (total_df['funding_rate_x'] - total_df['funding_rate_y']).abs()

In [69]:
total_df.sort_values(by=['funding_rate_diff'], ascending=False).head(20)

,symbol_x,funding_rate_x,funding_time_x,base_asset,quote_asset_x,datetime_now_x,market_code_x,symbol_y,funding_rate_y,funding_time_y,quote_asset_y,datetime_now_y,market_code_y,funding_rate_diff
32,GASUSDT,-0.020000,2023-11-10 08:00:00,GAS,USDT,2023-11-10 07:09:31.772,BINANCE_USD_M,GAS-USDT-SWAP,-0.008257,2023-11-10 08:00:00,USDT,2023-11-10 07:07:06.365,OKX_USD_M,0.011743
77,GAS-USDT-SWAP,-0.008257,2023-11-10 08:00:00,GAS,USDT,2023-11-10 07:07:06.365,OKX_USD_M,GASUSDT,-0.020000,2023-11-10 08:00:00,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M,0.011743
62,FITFI-USDT-SWAP,-0.001187,2023-11-10 08:00:00,FITFI,USDT,2023-11-10 07:07:06.365,OKX_USD_M,FITFIUSDT,0.001404,2023-11-10 08:00:00,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M,0.002591
18,BLUR-USDT-SWAP,-0.001278,2023-11-10 08:00:00,BLUR,USDT,2023-11-10 07:07:06.365,OKX_USD_M,BLURUSDT,0.001121,2023-11-10 08:00:00,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M,0.002399
95,BLURUSDT,0.000505,2023-11-10 08:00:00,BLUR,USDT,2023-11-10 07:09:31.772,BINANCE_USD_M,BLUR-USDT-SWAP,-0.001278,2023-11-10 08:00:00,USDT,2023-11-10 07:07:06.365,OKX_USD_M,0.001784
40,RDNT-USDT-SWAP,-0.001470,2023-11-10 08:00:00,RDNT,USDT,2023-11-10 07:07:06.365,OKX_USD_M,RDNTUSDT,0.000100,2023-11-10 08:00:00,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M,0.001570
94,APTUSDT,0.000100,2023-11-10 08:00:00,APT,USDT,2023-11-10 07:09:31.772,BINANCE_USD_M,APT-USDT-SWAP,-0.001306,2023-11-10 08:00:00,USDT,2023-11-10 07:07:06.365,OKX_USD_M,0.001406
7,APTUSD_PERP,0.000100,2023-11-10 08:00:00,APT,USD,2023-11-10 07:09:32.070,BINANCE_COIN_M,APT-USDT-SWAP,-0.001306,2023-11-10 08:00:00,USDT,2023-11-10 07:07:06.365,OKX_USD_M,0.001406
39,APT-USDT-SWAP,-0.001306,2023-11-10 08:00:00,APT,USDT,2023-11-10 07:07:06.365,OKX_USD_M,APTUSDT,0.000100,2023-11-10 08:00:00,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M,0.001406
121,GFT-USDT-SWAP,-0.001032,2023-11-10 08:00:00,GFT,USDT,2023-11-10 07:07:06.365,OKX_USD_M,GFTUSDT,0.000319,2023-11-10 08:00:00,USDT,2023-11-10 07:09:30.355,BYBIT_USD_M,0.001351
